# Neo4j Knowledge Graph & GraphRAG System - Complete Pipeline

This notebook provides a complete and verified pipeline for:
1. **Graph Construction**: Creating Table and Column nodes in Neo4j.
2. **Semantic Matching**: Identifying potential column relationships using ChromaDB and Gemini embeddings.
3. **Data-Driven Verification**: Performing actual inner joins on Datalake CSVs to verify relationships.
4. **RAG Integration**: Indexing the verified graph for vector search and providing a Q&A interface.

### Step 1: Install Dependencies
Run this cell if you haven't installed the required libraries.

In [ ]:
%pip install pandas neo4j chromadb google-genai tqdm numpy requests python-dotenv

### Step 2: Configuration & Environment Setup

In [ ]:
import sys
import os
import pandas as pd
from neo4j import GraphDatabase
import logging
import chromadb
from google import genai
import numpy as np
from tqdm.auto import tqdm
import ast
import time
from pathlib import Path

# --- Paths ---
CSV_PATH = '../Data_Dictionary/data_dictionary_enriched_v2.csv'
DATALAKE_PATH = r'i:\My Drive\HyTE' # Path to your source CSV files
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "data_dict1"
CHROMA_DB_PATH = "./chroma_db" # Path to save the vector database
FORCE_RECREATE_VDB = False      # Set to True to wipe and recreate embeddings

# --- Google Drive / Windows Quirk Fix ---
if not os.path.exists('.env'):
    try:
        with open('.env', 'w') as f: pass
    except: pass

# Add Data_Dictionary to path to import config
sys.path.append(os.path.abspath("../Data_Dictionary"))
try:
    from config import API_KEYS, MODELS
    import gemini_client
except ImportError:
    print("Warning: Could not import API_KEYS/gemini_client. Using placeholders.")
    API_KEYS = ["YOUR_API_KEY"]
    MODELS = ["gemini-1.5-flash"]

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

### Step 3: Initialize Clients

In [ ]:
# Initialize Neo4j Driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Initialize Gemini Client (for Q&A)
try:
    g_client = gemini_client.get_client()
except:
    g_client = None

# Initialize SDK Client for embeddings (with rotation logic)
current_key_index = 0
sdk_genai_client = genai.Client(api_key=API_KEYS[current_key_index])

# Initialize ChromaDB with Persistence
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
print(f"Clients initialized. Vector DB at: {CHROMA_DB_PATH}")

### Step 4: Helper Functions

In [ ]:
def get_embedding_with_rotation(text):
    """Fetches embeddings with automatic API key rotation on quota limits."""
    global current_key_index, sdk_genai_client
    while current_key_index < len(API_KEYS):
        try:
            result = sdk_genai_client.models.embed_content(
                model="gemini-embedding-001", 
                contents=text
            )
            return result.embeddings[0].values
        except Exception as e:
            err_str = str(e).lower()
            if "429" in err_str or "quota" in err_str or "limit" in err_str:
                current_key_index += 1
                if current_key_index < len(API_KEYS):
                    print(f"⚠️ Quota hit. Rotating to API Key #{current_key_index + 1}")
                    sdk_genai_client = genai.Client(api_key=API_KEYS[current_key_index])
                    continue
            raise e

def check_length_consistency(sample_str):
    """Checks if sample values have consistent string lengths (useful for IDs)."""
    try:
        samples = ast.literal_eval(sample_str)
        if not samples or not isinstance(samples, list): return False, 0
        lengths = [len(str(s)) for s in samples]
        if len(set(lengths)) == 1: return True, lengths[0]
        return False, 0
    except: return False, 0

def calculate_weighted_score(desc_sim, samp_sim, len_match):
    """Weighted similarity score: 0.6*Desc + 0.2*Length + 0.2*Sample"""
    m_desc = 1 # We assuming sim_desc is high enough to reach here
    m_samp = 1 if samp_sim >= 0.8 else 0
    m_len = 1 if len_match else 0
    return (0.6 * m_desc) + (0.2 * m_len) + (0.2 * m_samp)

def verify_join(t1, col1, t2, col2):
    """Verifies if two columns can actually join by checking local CSV files."""
    file1 = os.path.join(DATALAKE_PATH, f"{t1}.csv")
    file2 = os.path.join(DATALAKE_PATH, f"{t2}.csv")
    
    if not os.path.exists(file1) or not os.path.exists(file2):
        return False, 0
    
    try: 
        # Use usecols for speed
        df1 = pd.read_csv(file1, usecols=[col1])
        df2 = pd.read_csv(file2, usecols=[col2])
        common = set(df1[col1].unique()).intersection(set(df2[col2].unique()))
        common = {v for v in common if pd.notna(v)}
        return len(common) > 0, len(common)
    except: return False, 0

### Step 5: Base Knowledge Graph Construction

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from {CSV_PATH}")

def create_base_graph(tx, df):
    # Tables
    unique_tables = df[['Table Name', 'Table Description']].drop_duplicates('Table Name')
    for _, row in unique_tables.iterrows():
        tx.run("MERGE (t:Table {table_name: $name}) SET t.table_desc = $desc", 
               name=row['Table Name'], desc=row['Table Description'])
    
    # Columns
    for _, row in df.iterrows():
        tx.run("""
        MERGE (c:Column {table_name: $tname, column_name: $cname})
        SET c.col_desc = $cdesc, c.fill_rate = $fr, c.sample_vals = $sv
        WITH c
        MATCH (t:Table {table_name: $tname})
        MERGE (c)-[:`column within table`]->(t)
        """, tname=row['Table Name'], cname=row['Column Name'], 
            cdesc=row['Column Description'], fr=row['Fill Rate'], sv=row['Unique Values'])

with driver.session() as session:
    session.execute_write(create_base_graph, df)
print("Base graph (Tables & Columns) created.")

### Step 6: Sync Embeddings with ChromaDB

In [ ]:
if FORCE_RECREATE_VDB:
    try: chroma_client.delete_collection("descriptions")
    except: pass
    try: chroma_client.delete_collection("samples")
    except: pass

desc_coll = chroma_client.get_or_create_collection("descriptions", metadata={"hnsw:space": "cosine"})
samp_coll = chroma_client.get_or_create_collection("samples", metadata={"hnsw:space": "cosine"})

existing_desc_ids = set(desc_coll.get()['ids'])
items_to_embed = [row for _, row in df.iterrows() if f"{row['Table Name']}.{row['Column Name']}" not in existing_desc_ids]

if items_to_embed:
    print(f"Embedding {len(items_to_embed)} missing columns...")
    for row in tqdm(items_to_embed):
        col_id = f"{row['Table Name']}.{row['Column Name']}"
        desc_text = f"Column: {row['Column Name']}. Description: {row['Column Description']}"
        samp_text = f"Column: {row['Column Name']}. Samples: {row['Unique Values']}"
        
        desc_emb = get_embedding_with_rotation(desc_text)
        samp_emb = get_embedding_with_rotation(samp_text)
        
        desc_coll.add(ids=[col_id], embeddings=[desc_emb], metadatas=[{"table": row['Table Name'], "column": row['Column Name']}])
        samp_coll.add(ids=[col_id], embeddings=[samp_emb], metadatas=[{"table": row['Table Name'], "column": row['Column Name']}])
else:
    print("✅ All embeddings already exist in persistent storage.")

# Load into memory for matching cycle
all_desc = desc_coll.get(include=['embeddings'])
all_samp = samp_coll.get(include=['embeddings'])
cached_desc_embeddings = {id: emb for id, emb in zip(all_desc['ids'], all_desc['embeddings'])}
cached_samp_embeddings = {id: emb for id, emb in zip(all_samp['ids'], all_samp['embeddings'])}
print("Embeddings loaded into memory cache.")

### Step 7: Advanced Matching & Data-Driven Verification

In [ ]:
verified_results = []
processed_pairs = set()

print("Starting advanced matching with data-driven verification...")
for _, row_a in tqdm(df.iterrows(), total=len(df)):
    id_a = f"{row_a['Table Name']}.{row_a['Column Name']}"
    
    # Query for description similarity
    res = desc_coll.query(query_embeddings=[cached_desc_embeddings[id_a]], n_results=10)
    
    for i in range(len(res['ids'][0])):
        id_b = res['ids'][0][i]
        sim_desc = 1 - res['distances'][0][i]
        
        if id_a == id_b: continue
        pair = tuple(sorted([id_a, id_b]))
        if pair in processed_pairs: continue
        processed_pairs.add(pair)
        
        t1, c1 = id_a.split('.')
        t2, c2 = id_b.split('.')
        if t1 == t2: continue # Same table, usually not a relationship
        
        if sim_desc > 0.8:
            # 1. Sample Similarity
            sim_samp = 0
            samp_res = samp_coll.query(query_embeddings=[cached_samp_embeddings[id_a]], n_results=20)
            if id_b in samp_res['ids'][0]:
                id_b_idx = samp_res['ids'][0].index(id_b)
                sim_samp = 1 - samp_res['distances'][0][id_b_idx]
            
            # 2. Length Consistency
            cons_a, len_a = check_length_consistency(row_a['Unique Values'])
            row_b = df[(df['Table Name'] == t2) & (df['Column Name'] == c2)].iloc[0]
            cons_b, len_b = check_length_consistency(row_b['Unique Values'])
            len_match = (cons_a and cons_b and len_a == len_b)
            
            # 3. Weighted Score
            score = calculate_weighted_score(sim_desc, sim_samp, len_match)
            
            if score > 0.7:
                # 4. DATA-DRIVEN VERIFICATION (The Critical Step)
                is_valid, common_count = verify_join(t1, c1, t2, c2)
                if is_valid:
                    verified_results.append({"source": id_a, "target": id_b, "score": score * 100})

print(f"Found {len(verified_results)} verified relationships.")

### Step 8: Finalize Graph (Relationships & RAG Index)

In [ ]:
with driver.session() as session:
    # 1. Ingest Relationships
    session.run("""
    UNWIND $batch AS row
    MATCH (c1:Column {table_name: split(row.source, '.')[0], column_name: split(row.source, '.')[1]})
    MATCH (c2:Column {table_name: split(row.target, '.')[0], column_name: split(row.target, '.')[1]})
    MERGE (c1)-[r:potentially_same_column]-(c2)
    SET r.confidence_score = row.score
    """, batch=verified_results)

    # 2. Enrich nodes with RAG Embeddings
    print("Enriching nodes with RAG embeddings...")
    records = session.run("MATCH (c:Column) RETURN c.table_name as t, c.column_name as n, c.col_desc as d, c.sample_vals as s")
    for r in records:
        content = f"Table: {r['t']}. Column: {r['n']}. Description: {r['d']}. Samples: {r['s']}"
        session.run("MATCH (c:Column {table_name: $t, column_name: $n}) SET c.embedding = $emb", 
                    t=r['t'], n=r['n'], emb=get_embedding_with_rotation(content))

    # 3. Create Vector Index
    test_emb = get_embedding_with_rotation("dimension check")
    dim = len(test_emb)
    try: session.run("DROP INDEX column_vector_index")
    except: pass
    session.run("""
    CREATE VECTOR INDEX column_vector_index FOR (c:Column) ON (c.embedding)
    OPTIONS {indexConfig: {`vector.dimensions`: $dim, `vector.similarity_function`: 'cosine'}}
    """, dim=dim)

print("🚀 Pipeline Complete! Knowledge Graph is verified and indexed.")

### Step 9: GraphRAG Interface

In [ ]:
class GraphRAGRetriever:
    def __init__(self, driver):
        self.driver = driver

    def retrieve(self, query, top_k=5):
        query_emb = get_embedding_with_rotation(query)
        cypher_query = """
        CALL db.index.vector.queryNodes('column_vector_index', $k, $emb) 
        YIELD node AS col, score
        MATCH (col)-[:`column within table`]->(t:Table)
        OPTIONAL MATCH (col)-[r:potentially_same_column]-(other:Column)
        RETURN col.column_name as name, col.col_desc as desc, col.sample_vals as samples, 
               t.table_name as table, t.table_desc as table_desc, score,
               collect(DISTINCT other.table_name + '.' + other.column_name) as related_cols
        """
        with self.driver.session() as session:
            results = session.run(cypher_query, emb=query_emb, k=top_k)
            context_items = []
            for res in results:
                item = f"Table: {res['table']} ({res['table_desc']})\n"
                item += f"Column: {res['name']}\nDescription: {res['desc']}\nSamples: {res['samples']}\n"
                if res['related_cols']:
                    item += f"Verified Related columns: {', '.join(res['related_cols'])}\n"
                context_items.append(item)
            return "\n---\n".join(context_items)

def ask_graph(question):
    retriever = GraphRAGRetriever(driver)
    context = retriever.retrieve(question)
    
    prompt = f"""
    You are a data analyst. Use the Knowledge Graph context provided below to answer the user's question.
    The relationships in this context have been verified against actual data joins.

    Context:
    {context}

    Question: {question}
    
    Provide an accurate, concise, and professional answer.
    """
    
    if gemini_client:
        answer = gemini_client.call_gemini(prompt)
        print(f"\nAnswer: {answer}")
    else:
        print("\nGemini client not available to generate answer. Context retrieved:")
        print(context)

print("GraphRAG Interface Ready.")

### Test the System

In [ ]:
question = "Which tables contain customer identification information and how do they relate?"
ask_graph(question)